# LongFlow — teacher-polish knob + combined clean pool

Runtime: **A100 GPU**, ~1.5–2 h. Pre-registered criteria: NOTES.md
"POLISH + COMBINED-POOL NIGHT PRE-REGISTRATION" (2026-08-18).

Two no-param quality levers in one night:
- **Arm P (the knob):** flow-sample a latent, re-noise it slightly, let the
  frozen TEACHER DDPM head run its last k steps to snap it back onto the
  decoder's manifold. k ∈ {0,1,2,3,5} — ten teacher-forced wavs for Josh's
  ear. Zero training.
- **Arm M (combined pool):** v1 clean short-clips + v2 clean long-form =
  ~746K frames, control architecture, 20K steps.
- **Arm CL:** closed loop for the combined head (2 seeds) + polish k=2 on
  both heads.

| cell | what |
|---|---|
| 1 | cold start (+ bulk-copy BOTH caches local) |
| 2 | pools: v2 clean frames + v1 frames → combined |
| 3 | polish helper + ARM P renders (knob wavs → Drive; listen anytime) |
| 4 | ARM M training (20K, ckpts every 5K) |
| 5 | held-out teacher-forced at 20K (heun8+CFG) |
| 6 | ARM CL closed loop (4 renders) |
| 7 | bundle → Drive root `polish_eval.zip` |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Polish+Combined v1.0 (2026-08-18)"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed — check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, _CFGField
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import (
    PairData, filter_flagged, load_checkpoint, pairs_from_files, train,
)

CACHE_V2_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v2"
CACHE_V1_DRIVE = "/content/drive/MyDrive/longflow_p1_cache"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
OUT = "/content/polish"
DRIVE_OUT = "/content/drive/MyDrive/longflow_polish"
EVAL_DIR = "/content/polish_eval"
for d in (OUT, DRIVE_OUT, EVAL_DIR, f"{DRIVE_OUT}/knob"):
    os.makedirs(d, exist_ok=True)

LOCAL_V2 = "/content/cache_v2"
if not os.path.exists(LOCAL_V2) or len(glob.glob(f"{LOCAL_V2}/*.pt")) < 240:
    os.makedirs(LOCAL_V2, exist_ok=True)
    print("bulk-copying cache v2 (~4 GB)...", flush=True)
    !cp {CACHE_V2_DRIVE}/*.pt {LOCAL_V2}/
print(f"{len(glob.glob(f'{LOCAL_V2}/*.pt'))} v2 files local")

LOCAL_V1 = "/content/cache_v1"
if not os.path.exists(LOCAL_V1) or len(glob.glob(f"{LOCAL_V1}/*.pt")) < 9000:
    os.makedirs(LOCAL_V1, exist_ok=True)
    print("bulk-copying v1 cache (~10K files, several minutes — one cp, not per-file reads)...", flush=True)
    !cp {CACHE_V1_DRIVE}/*.pt {LOCAL_V1}/ 2>/dev/null || true
print(f"{len(glob.glob(f'{LOCAL_V1}/*.pt'))} v1 files local")

if os.path.exists(f"{DRIVE_OUT}/polish_report.json"):
    with open(f"{DRIVE_OUT}/polish_report.json") as f:
        report = json.load(f)
    print(f"resuming: {len(report['runs'])} runs already recorded")
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta, sub=""):
    d = f"{DRIVE_OUT}/{sub}" if sub else DRIVE_OUT
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{d}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/polish_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def decode_latents(z, chunk_frames=225):
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt {tuple(fn(chunk).shape)} failed: {repr(e)[:150]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed — paste the errors to Claude")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

print("READY")


In [ ]:
# ===== Pools: v2 clean frames + v1 frames -> combined (all-clean, both registers) =====
HELD_OUT_PER_BIN = 5
FLAGS = "/content/LongFlow/experiments/p1_flow_head/capture_v2_audit_flags.json"

v2_files = filter_flagged(sorted(glob.glob(f"{LOCAL_V2}/*.pt")), FLAGS)

def fname_bin(path):
    return int(Path(path).stem.split("_")[1].rstrip("w"))

by_bin = {}
for f in v2_files:
    by_bin.setdefault(fname_bin(f), []).append(f)
held_out_by_bin = {b: fs[:HELD_OUT_PER_BIN] for b, fs in sorted(by_bin.items())}
held_out_files = [f for fs in held_out_by_bin.values() for f in fs]
train_files_v2 = [f for b, fs in sorted(by_bin.items()) for f in fs[HELD_OUT_PER_BIN:]]
print(f"v2: held-out {len(held_out_files)} / train {len(train_files_v2)} (identical split to HV2/cleanabl)")

full = pairs_from_files(train_files_v2, dual_stream=True)
mask = full.sigma_bucket == 0
v2_hidden = full.hidden[mask]
v2_latent_raw = (full.latent * full.std + full.mean)[mask]
print(f"v2 clean frames: {v2_hidden.shape[0]}")
del full

t0 = time.time()
v1_files = sorted(glob.glob(f"{LOCAL_V1}/*.pt"))
v1_h, v1_l = [], []
for i, f in enumerate(v1_files):
    try:
        utt = load_utterance(f)
    except Exception as e:
        print(f"skip {Path(f).name}: {repr(e)[:100]}")
        continue
    v1_h.append(utt.hidden.float())
    v1_l.append(utt.latent.float())
    if (i + 1) % 2000 == 0:
        print(f"  {i+1}/{len(v1_files)} v1 files loaded ({time.time()-t0:.0f}s)", flush=True)
v1_hidden = torch.cat(v1_h)
v1_latent_raw = torch.cat(v1_l)
del v1_h, v1_l
print(f"v1 frames: {v1_hidden.shape[0]}  ({time.time()-t0:.0f}s)")

hidden = torch.cat([v1_hidden, v2_hidden])
latent_raw = torch.cat([v1_latent_raw, v2_latent_raw])
mean = latent_raw.mean(dim=0)
std = latent_raw.std(dim=0).clamp_min(1e-4)
data = PairData(hidden=hidden, latent=(latent_raw - mean) / std, mean=mean, std=std)
print(f"COMBINED pool: {data.hidden.shape[0]} frames  d_model={data.d_model}  d_latent={data.d_latent}")
del v1_hidden, v1_latent_raw, v2_hidden, v2_latent_raw, hidden, latent_raw


## 3. ARM P — the knob (teacher-forced; listen from Drive any time)

`Drive/longflow_polish/knob/` fills with `<utt>_k{0,1,2,3,5}.wav` +
`<utt>_teacher.wav`. k=0 is the raw cleanabl head; each higher k hands the
latent to the frozen teacher head for its last k denoise steps. **This is
the quarter-turn — listen in order.**


In [ ]:
def polish(z, cond, neg, k, cfg_scale=1.3, total_steps=10, seed=None):
    """z [T, d_latent] head-space -> teacher-polished latents (same space).
    Mirrors sample_speech_tokens' own loop (vendored DPMSolverMultistep,
    in-loop CFG) but starts from the student latent re-noised at the k-th-
    from-last timestep instead of pure noise. k=0 -> passthrough."""
    if k <= 0:
        return z
    sched = model.model.noise_scheduler
    sched.set_timesteps(total_steps)  # fresh state per call, same as the teacher's own usage
    ts = sched.timesteps[-k:]
    zt = z.to("cuda", torch.bfloat16)
    cond2 = torch.cat([cond, neg], dim=0).to("cuda", torch.bfloat16)
    g = None if seed is None else torch.Generator(device="cuda").manual_seed(seed)
    noise = torch.randn(zt.shape, device="cuda", dtype=torch.float32, generator=g).to(torch.bfloat16)
    zt = sched.add_noise(zt, noise, ts[0].expand(zt.shape[0]))
    for t in ts:
        combined = torch.cat([zt, zt], dim=0)
        eps = model.model.prediction_head(
            combined, t.repeat(combined.shape[0]).to(combined), condition=cond2)
        c_eps, u_eps = torch.split(eps, len(eps) // 2, dim=0)
        guided = u_eps + cfg_scale * (c_eps - u_eps)
        zt = sched.step(guided, t, zt).prev_sample
    return zt.float()

def flow_cfg_sample(head, mean_t, std_t, utt, seed=0):
    field = _CFGField(head, utt.neg_hidden.float().cuda(), 1.3)
    g = torch.Generator(device="cuda").manual_seed(seed)
    z = heun_sample(field, utt.hidden.float().cuda(), head.cfg.d_latent,
                    nfe=8, sway=0.0, generator=g)
    return z * std_t.cuda() + mean_t.cuda()

head_cl, mean_cl, std_cl = load_checkpoint(f"{CKPT_DIR}/cleanabl_20k_step20000.pt")
head_cl = head_cl.to("cuda")

K_VALUES = [0, 1, 2, 3, 5]
knob_utts = [held_out_by_bin[150][0], held_out_by_bin[1200][0]]
knob_manifest = {}
for fpath in knob_utts:
    utt = load_utterance(fpath)
    tpath = f"{DRIVE_OUT}/knob/{utt.utt_id}_teacher.wav"
    if not os.path.exists(tpath):
        wav_t = decode_latents(utt.latent.float())
        sf.write(tpath, wav_t, 24000)
        sf.write(f"{OUT}/{utt.utt_id}_teacher.wav", wav_t, 24000)
    z0 = flow_cfg_sample(head_cl, mean_cl, std_cl, utt, seed=0)
    entries = {}
    for k in K_VALUES:
        tag = f"{utt.utt_id}_k{k}"
        wpath = f"{DRIVE_OUT}/knob/{tag}.wav"
        if not os.path.exists(wpath):
            zk = polish(z0.clone(), utt.hidden.float().cuda(),
                        utt.neg_hidden.float().cuda(), k, seed=0)
            wav = decode_latents(zk)
            sf.write(wpath, wav, 24000)
            sf.write(f"{OUT}/{tag}.wav", wav, 24000)
        entries[k] = f"{tag}.wav"
        print(f"knob: {tag} done", flush=True)
    knob_manifest[utt.utt_id] = {"teacher": f"{utt.utt_id}_teacher.wav",
                                 "text": utt.text, "k_wavs": entries,
                                 "target_words": fname_bin(fpath)}
report["knob"] = knob_manifest
with open(f"{DRIVE_OUT}/polish_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("ARM P done — knob wavs on Drive; listen whenever (k0 -> k5, vs teacher)")


In [ ]:
# ===== ARM M — combined-pool training: 20K, ckpts every 5K =====
TAG = "combined_20k"
final = f"{CKPT_DIR}/{TAG}_step20000.pt"
if os.path.exists(final):
    print(f"{TAG}: final checkpoint already on Drive — skipping")
else:
    head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent))
    print(f"head: {head.param_count()/1e6:.2f}M params (control architecture)")
    t0 = time.time()
    train(head, data, steps=20000, batch_size=1024, lr=2e-4, lr_final=2e-5,
          ema_decay=0.9999, device="cuda", log_every=1000,
          checkpoint_every=5000,
          checkpoint_path_fn=lambda s: f"{CKPT_DIR}/{TAG}_step{s}.pt")
    print(f"done in {(time.time()-t0)/60:.1f} min")
    del head
    torch.cuda.empty_cache()


In [ ]:
# ===== Held-out teacher-forced at 20K (heun8+CFG, n=25) =====
manifest = {"held_out_per_bin": HELD_OUT_PER_BIN, "teacher": {}, "checkpoints": {}}
head_m, mean_m, std_m = load_checkpoint(f"{CKPT_DIR}/combined_20k_step20000.pt")
head_m = head_m.to("cuda")

entries = []
for fpath in held_out_files:
    utt = load_utterance(fpath)
    tname = f"{utt.utt_id}_teacher.wav"
    if not os.path.exists(f"{EVAL_DIR}/{tname}"):
        sf.write(f"{EVAL_DIR}/{tname}", decode_latents(utt.latent.float()), 24000)
    manifest["teacher"][utt.utt_id] = {"audio": tname, "text": utt.text,
                                       "target_words": fname_bin(fpath)}
    name = f"{utt.utt_id}_step20000_M.wav"
    if not os.path.exists(f"{EVAL_DIR}/{name}"):
        z = flow_cfg_sample(head_m, mean_m, std_m, utt, seed=0)
        sf.write(f"{EVAL_DIR}/{name}", decode_latents(z), 24000)
    entries.append({"utt_id": utt.utt_id, "audio": name, "teacher_audio": tname,
                    "text": utt.text, "target_words": fname_bin(fpath), "arm": "M"})
    print(f"rendered {utt.utt_id}", flush=True)
manifest["checkpoints"]["20000:M"] = entries
with open(f"{EVAL_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("held-out eval rendered")


## 6. ARM CL — closed loop (same GN5–8 protocol)

Four renders: combined head seeds 0/1; polish k=2 on the cleanabl head and
on the combined head (seed 0). The polish patch snaps every frame's latent
back to the teacher manifold before it re-enters the loop — the on-manifold
feedback hypothesis gets its first closed-loop test here.


In [ ]:
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern}")

sents = []
for f in sorted(glob.glob(f"{LOCAL_V1}/*.pt"))[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("closed-loop script:", w, "words")
report["cl_script"] = ABL_SCRIPT
report["cl_words"] = w

class PolishCFGPatch(CFGFlowHeadPatch):
    """CFGFlowHeadPatch + teacher-polish of each frame's latent before it
    re-enters the loop (and the audio). k = polish steps."""

    def __init__(self, *args, polish_k=2, **kwargs):
        super().__init__(*args, **kwargs)
        self.polish_k = polish_k

    def __enter__(self):
        super().__enter__()
        inner = self.model.sample_speech_tokens
        patch = self

        def flow_sample_polished(condition, neg_condition=None, cfg_scale=None):
            z = inner(condition, neg_condition=neg_condition, cfg_scale=cfg_scale)
            if neg_condition is not None and patch.polish_k > 0:
                z2 = polish(z.float(), condition.float().cuda(),
                            neg_condition.float().cuda(), patch.polish_k)
                patch.latents[-1] = z2.detach().float().cpu()  # record the polished latent
                return z2.to(condition.dtype)
            return z

        self.model.sample_speech_tokens = flow_sample_polished
        return self

CL_ARMS = [  # (tag, head, mean, std, seed, polish_k)
    ("combined_cfg_heun8_s0", "M", 0, 0),
    ("combined_cfg_heun8_s1", "M", 1, 0),
    ("cleanabl_polish2_s0", "C", 0, 2),
    ("combined_polish2_s0", "M", 0, 2),
]
HEADS = {"M": (head_m, mean_m, std_m), "C": (head_cl, mean_cl, std_cl)}
for tag, hk, seed, pk in CL_ARMS:
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    h, mn, sd = HEADS[hk]
    torch.manual_seed(seed)
    cls = (lambda *a, **k: PolishCFGPatch(*a, polish_k=pk, **k)) if pk > 0 else CFGFlowHeadPatch
    with cls(model, h, mn, sd, nfe=8, sway=0.0, sampler=heun_sample) as patch, \
         torch.inference_mode():
        gen = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    save_wav(tag, wav, {"head": hk, "seed": seed, "polish_k": pk,
                        "frames": patch.calls,
                        "latent_std": round(float(zs.std()), 3)})


In [ ]:
# ===== Bundle -> Drive root =====
import zipfile
teacher_ref = f"{GATE3_DIR}/t1_turnsplit_p0.wav"
assert os.path.exists(teacher_ref), "GN3 teacher reference missing from Drive"
shutil.copy(teacher_ref, f"{EVAL_DIR}/t1_turnsplit_p0.wav")
with open(f"{EVAL_DIR}/polish_report.json", "w") as f:
    json.dump(report, f, indent=2)

ZIP = "/content/drive/MyDrive/polish_eval.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    for f in os.listdir(EVAL_DIR):
        z.write(f"{EVAL_DIR}/{f}", f)
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
    for f in os.listdir(f"{DRIVE_OUT}/knob"):
        z.write(f"{DRIVE_OUT}/knob/{f}", f"knob/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e9:.2f} GB) — run score_polish_gpu_colab.ipynb next")
